# N018 · 矩阵遍历与边界模拟

**目标：** 将二维坐标变化和边界收缩写成无重复遍历。

**先修：** N005, N017。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 行列索引；四方向；转置；螺旋边界；一行一列退化情形。

**配套讲解来源：** [同名逐章意见](../../comment/018_matrix_traversal_simulation.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章处理两个经典矩阵题。第一个是**螺旋遍历**：给你一个二维矩阵，从左上角出发，沿顺时针方向一圈一圈往里走，把沿途的元素按访问顺序列成一个一维数组。第二个是**方阵顺时针旋转 90 度**：把一个 n×n 的方阵在原地转四分之一圈，不许开新矩阵。

两个具体到数字的例子（都用 “运行示例”部分 里的 `m=[[1,2,3],[4,5,6],[7,8,9]]`）：

- 输入这个 3×3 矩阵 → 螺旋输出 `[1,2,3,6,9,8,7,4,5]` → 为什么？先走第一行 1,2,3；到头后右列向下走 6,9；底行向左走 8,7；左列向上走 4；最后走进中心拿到 5。
- 同一个矩阵顺时针旋转 90 度 → 得到 `[[7,4,1],[8,5,2],[9,6,3]]` → 为什么？第一列 1,4,7 竖着读出来倒着放就是新第一行 7,4,1，其余同理。

这两题的共同考点是"坐标和边界"：你要时刻清楚自己身在哪个格子、下一步往哪走、什么时候该拐弯、什么时候已经没路了。

## 2. 基础知识：先读懂这些词

- **行列索引（row/column index）**：矩阵里一个格子的地址 `matrix[i][j]`，i 是行号（从上往下数），j 是列号（从左往右数）。Python 里矩阵就是"列表的列表"，先取第 i 行、再取该行第 j 列。
- **四方向（four directions）**：螺旋每圈要依次走上（顶行，向右）、右（右列，向下）、下（底行，向左）、左（左列，向上）四条边，方向依次为右→下→左→上。
- **边界收缩（boundary shrinking）**：用四个变量 top、bottom、left、right 圈出"还没访问过的区域"。每处理完一条边，就把对应的边界往里推一格（比如处理完顶行就 top+=1），表示那一行/列已经退役。
- **单行单列退化情形**：当未访问区域只剩一行或一列时，"走完顶行"其实已经把这一层全部元素走完了，如果照常再去走底行、左列，同一批元素就会被输出两次。这是螺旋题最经典的坑，必须加判断。
- **转置（transpose）**：把矩阵沿主对角线（左上到右下）翻面，坐标 `(i,j)` 变成 `(j,i)`，即原来的第 i 行第 j 列变成第 j 行第 i 列。
- **每行反转（reverse each row）**：把每一行内部的元素左右颠倒。
- **旋转的坐标映射**：顺时针转 90 度后，原坐标 `(i,j)` 的元素会落在 `(j, n-1-i)`。它恰好等于"先转置再每行反转"这两步的复合——这是旋转题的巧妙分解。

## 3. 思路推导：从小例子开始

### 螺旋遍历的思路

- **Step 1：圈住未访问区。** 用 top、bottom、left、right 四个变量框出当前还没走过的矩形区域，初始分别是 0、rows-1、0、cols-1。
- **Step 2：一次走四条边。** 按顺时针顺序：顶行从左到右走一遍；右列从上到下走一遍；底行从右到左走一遍；左列从下到上走一遍。每走完一条边立刻收缩对应边界。
- **Step 3：走对边之前先检查还剩不剩。** 走完顶行和右列之后，剩下的区域可能只剩一行了；此时底行其实已经没了（或就是刚走过的内容），必须先问一句"top 还 ≤ bottom 吗"，是才走底行。同理左列走之前要问"left 还 ≤ right 吗"。这两句检查就是防重复的关键。
- **Step 4：循环直到区域空。** 只要 top>bottom 或 left>right，说明所有元素都已访问，返回结果。

**如果没有那两句 if 检查会发生什么**（用单行矩阵 `[[1,2,3]]` 演示）：顶行走完输出 1,2,3，top 变 1、right 变 1。此时若不加检查直接走"底行"，会输出 matrix[0][1]=2、matrix[0][0]=1，总输出变成 `[1,2,3,2,1]`——同一批元素被收了两遍。加上 `top<=bottom` 的检查后，这一步被跳过，输出正确地停在 `[1,2,3]`。

**手算演示**（3×3 矩阵，初始 top=0,bottom=2,left=0,right=2）：

| 阶段 | 走的边 | 输出 | 收缩后边界 |
|---|---|---|---|
| 第 1 轮-顶行 | `matrix[0][0..2]` | 1,2,3 | top=1 |
| 第 1 轮-右列 | `matrix[1..2][2]` | 6,9 | right=1 |
| 第 1 轮-底行 | `matrix[2][1..0]` | 8,7 | bottom=1 |
| 第 1 轮-左列 | `matrix[1..1][0]` | 4 | left=1 |
| 第 2 轮-顶行 | `matrix[1][1..1]` | 5 | top=2（循环条件 top≤bottom 失败，结束） |

最终输出 `[1,2,3,6,9,8,7,4,5]`，与 “运行示例”部分 表格逐项一致。

### 方阵旋转的思路

- **Step 1：先转置。** 沿主对角线对称交换：遍历对角线右上方的每对 `(i,j)`，交换 `matrix[i][j]` 与 `matrix[j][i]`。转置的坐标效果是 `(i,j)→(j,i)`。
- **Step 2：再每行反转。** 每行左右翻转后，行内坐标 j 变成 n-1-j。两步复合：`(i,j) → (j,i) → (j,n-1-i)`，正是顺时针旋转 90 度的目标映射。

**手算演示**（同一个 3×3 矩阵）：转置后得到 `[[1,4,7],[2,5,8],[3,6,9]]`；每行反转后得到 `[[7,4,1],[8,5,2],[9,6,3]]`。验证一下映射：元素 1 原在 (0,0)，按公式应落到 (0, 3-1-0)=(0,2)，新矩阵右上角正是 1。

## 4. 核心代码：spiral_order

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def spiral_order(matrix):
    if not matrix or not matrix[0]:
        return []
    top, bottom, left, right = (0, len(matrix) - 1, 0, len(matrix[0]) - 1)
    out = []
    while top <= bottom and left <= right:
        for j in range(left, right + 1):
            out.append(matrix[top][j])
        top += 1
        for i in range(top, bottom + 1):
            out.append(matrix[i][right])
        right -= 1
        if top <= bottom:
            for j in range(right, left - 1, -1):
                out.append(matrix[bottom][j])
            bottom -= 1
        if left <= right:
            for i in range(bottom, top - 1, -1):
                out.append(matrix[i][left])
            left += 1
    return out
```

### 逐段读懂代码

### 函数 1：`spiral_order`（螺旋遍历）

先看入口与边界初始化：

```python
def spiral_order(matrix):
    if not matrix or not matrix[0]:
        return []
    top, bottom, left, right = (0, len(matrix) - 1, 0, len(matrix[0]) - 1)
    out = []
```

- `if not matrix or not matrix[0]: return []`：单行 if 的紧凑写法，挡住两种非法/退化输入——整个矩阵为空 `[]`，或第一行就是空 `[[]]`。这两种情况都没有元素可走，直接返回空列表，后面代码也不用再操心除零、越界。
- `top,bottom,left,right=...`：一行多赋值，把四个边界初始化为整个矩阵的四个边。
- `out=[]`：准备一个空列表，按访问顺序收集元素。

再看主循环（四条边、逐边收缩、防重复检查）：

```python
while top<=bottom and left<=right:
        for j in range(left,right+1): out.append(matrix[top][j])
        top+=1
        for i in range(top,bottom+1): out.append(matrix[i][right])
        right-=1
        if top<=bottom:
            for j in range(right,left-1,-1): out.append(matrix[bottom][j])
            bottom-=1
        if left<=right:
            for i in range(bottom,top-1,-1): out.append(matrix[i][left])
            left+=1
    return out
```

- `while top<=bottom and left<=right:`：只要未访问矩形还存在就继续；两个条件任一失败即全部访问完。
- 顶行 `for j in range(left,right+1): out.append(matrix[top][j])`：行号固定为 top，列号从 left 走到 right（含两端，所以 +1）。走完 `top+=1` 退役这行。
- 右列 `for i in range(top,bottom+1): out.append(matrix[i][right])`：列号固定 right，行号从**新的 top**（注意顶行刚收缩过）走到 bottom。走完 `right-=1`。
- `if top<=bottom:` 加底行：这句是防重复的第一道闸。设想只剩一行时，top 行走完后 top 已经越过 bottom，若不检查还会把"底行"（其实还是同一行）再输出一遍。`range(right,left-1,-1)` 是从 right 倒着走到 left（含 left，所以终点写 left-1）。走完 `bottom-=1`。
- `if left<=right:` 加左列：同理，只剩一列时右列走完后 left 已越过 right，这句防止左列重复输出。`range(bottom,top-1,-1)` 从 bottom 倒着走到 top。走完 `left+=1`。
- `return out`：返回按访问顺序收集的全部元素。

### 函数 2：`rotate_square`（方阵顺时针旋转 90 度，原地）

```python
def rotate_square(matrix):
    n = len(matrix)
    if any((len(row) != n for row in matrix)):
        raise ValueError('requires square matrix')
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = (matrix[j][i], matrix[i][j])
    for row in matrix:
        row.reverse()
```

- `if any(len(row)!=n for row in matrix): raise ValueError(...)`：前置检查——只要有一行的长度不等于行数，它就不是方阵，直接报错。因为非方阵的行数≠列数，原地旋转后形状对不上，这套交换逻辑根本无从谈起（练习 2 会让你说清楚原因）。
- 双层循环 `for i ... for j in range(i+1,n)`：只枚举主对角线**右上**一半的坐标对。若 j 也从 0 起，同一对 `(i,j)`、`(j,i)` 会被交换两次、等于没换；从 i+1 起恰好每对只交换一次。
- `matrix[i][j],matrix[j][i]=matrix[j][i],matrix[i][j]`：Python 的多重赋值，一步完成两个格子的互换，不需要临时变量。
- `for row in matrix: row.reverse()`：把每一行原地左右反转，完成"转置 + 行反转 = 顺时针 90 度"的第二步。整个过程没有新建矩阵，全是原地操作。

## 5. 分段实现：按顺序运行

**本章接口：** `spiral_order(matrix)`、`rotate_square(matrix)`

### spiral_order

In [1]:
def spiral_order(matrix):
    if not matrix or not matrix[0]:
        return []
    top, bottom, left, right = (0, len(matrix) - 1, 0, len(matrix[0]) - 1)
    out = []
    while top <= bottom and left <= right:
        for j in range(left, right + 1):
            out.append(matrix[top][j])
        top += 1
        for i in range(top, bottom + 1):
            out.append(matrix[i][right])
        right -= 1
        if top <= bottom:
            for j in range(right, left - 1, -1):
                out.append(matrix[bottom][j])
            bottom -= 1
        if left <= right:
            for i in range(bottom, top - 1, -1):
                out.append(matrix[i][left])
            left += 1
    return out

### rotate_square

In [2]:
def rotate_square(matrix):
    n = len(matrix)
    if any((len(row) != n for row in matrix)):
        raise ValueError('requires square matrix')
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = (matrix[j][i], matrix[i][j])
    for row in matrix:
        row.reverse()

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

m=[[1,2,3],[4,5,6],[7,8,9]]
order=spiral_order(m)
show_table(['访问序号','元素'], list(enumerate(order,1)))
rotate_square(m)
show_table(['旋转后行号','行'],list(enumerate(m)))

访问序号,元素
1,1
2,2
3,3
4,6
5,9
6,8
7,7
8,4
9,5


旋转后行号,行
0,"[7, 4, 1]"
1,"[8, 5, 2]"
2,"[9, 6, 3]"


## 7. 正确性、适用条件与复杂度

未访问元素始终位于当前闭矩形内；每次只取边界并将其删除。转置和反转的坐标复合直接给出旋转映射，且交换是双射。

**代价：** 螺旋 O(rows×cols) 时间与输出空间；旋转 O(n²) 时间、O(1) 辅助空间。

### 展开理解

**为什么是对的：** 先说螺旋。我们随时维护着一个"还没访问过的矩形"，由 top、bottom、left、right 四条边界围成。每条边走完就推掉一条边界，相当于把这一层"剥皮"丢掉，所以任何元素都不可能被访问两次——它要么在还没剥到的层里，要么已经随某条边退役了。那"一个不漏"呢？只要矩形还非空（top≤bottom 且 left≤right），循环就会继续剥，直到矩形彻底消失，说明每个格子都随某条边被收进 `out` 了。再说那两个 if 检查为什么必要：剥到只剩一行时，走完顶行这一行就退役了，但按流程接下来还要"走底行"，而此时底行边界与顶行边界已经交错，继续走就会把同一批元素再收一遍；检查 top≤bottom 就是在问"底行还真实存在吗"。旋转这边：转置沿对角线两两交换，每一对只换一次，交换本身是一一对应的（A 换到 B 的位置、B 换到 A 的位置），所以转置是精确的对角翻面；行反转也是精确的左右翻。两步复合的坐标效果 `(i,j)→(j,n-1-i)` 正是顺时针 90 度的定义式，所以结果必然正确。

**复杂度是多少：** 螺旋遍历每个元素恰好被访问一次，时间是 O(行数×列数)，输出的列表本身也这么大。旋转的转置要交换约 n²/2 对元素、行反转再动 n²/2 对，总计 O(n²) 时间，只用了常数个下标变量，辅助空间 O(1)。举个具体的数：500×500 的方阵有 25 万个元素，O(n²) 也就是五十万次左右的交换操作，一瞬间完成。

**两个常见疑问，顺便说清：**

- 疑问一："右列的行号为什么从新的 top 起步？"——因为顶行刚被 `top+=1` 收缩掉了。若还从旧 top 起，右上角那个元素会被输出两遍（顶行末尾一次、右列开头一次）。
- 疑问二："为什么旋转要检查方阵、螺旋却不用？"——螺旋对任意 rows×cols 都成立，四个边界天然适配长方形；旋转的"转置+行反转"依赖行列等长（交换 `matrix[i][j]` 与 `matrix[j][i]` 时两个位置必须都存在），所以入口必须先验方阵身份。

## 8. 单元测试：每个用例在检查什么

```python
assert spiral_order([[1, 2, 3], [4, 5, 6]]) == [1, 2, 3, 6, 5, 4]

assert spiral_order([[1, 2, 3]]) == [1, 2, 3]

assert spiral_order([[1], [2], [3]]) == [1, 2, 3]

assert spiral_order([]) == []

for n in range(5):
    m = [[i * n + j for j in range(n)] for i in range(n)]
    orig = [r[:] for r in m]
    for _ in range(4):
        rotate_square(m)
    assert m == orig

print('N018: 所有本章断言通过')
```

- 第 1 条是**宽矩阵正常值测试**（2×3）：`[[1,2,3],[4,5,6]]` 期望 `[1,2,3,6,5,4]`。这个输入只剩一行时会触发 top/bottom 检查（走完顶行 1,2,3 和右列 6 后只剩一行，靠检查避免重复），是检验防重复逻辑的主力用例。
- 第 2 条是**单行矩阵测试**（边界值）：一行三列从头走到尾就是答案，绝不允许输出两遍。这正是"单行退化"的直接测试。
- 第 3 条是**单列矩阵测试**（边界值）：三行一列向下走即可，对应"单列退化"，检验的是 left/right 检查那一侧。
- 第 4 条是**空矩阵测试**：输入 `[]` 应返回 `[]`，验证函数开头那句紧凑 if 挡非法输入的行为。
- 最后一段 for 是**旋转的组合律测试**：对 0×0 到 4×4 的每个方阵（元素值用 `i*n+j` 编码位置，方便比对），先复制一份 `orig`，然后连转 4 次。顺时针转 4 次共 360 度，矩阵必须回到原样，所以断言 `m==orig`。`orig=[r[:] for r in m]` 是浅拷贝每一行、留一份原始快照。转 4 次等于没转，这个性质比单次旋转的结果更难被"碰巧"满足，是很有力的正确性抽查。

In [4]:
assert spiral_order([[1, 2, 3], [4, 5, 6]]) == [1, 2, 3, 6, 5, 4]

assert spiral_order([[1, 2, 3]]) == [1, 2, 3]

assert spiral_order([[1], [2], [3]]) == [1, 2, 3]

assert spiral_order([]) == []

for n in range(5):
    m = [[i * n + j for j in range(n)] for i in range(n)]
    orig = [r[:] for r in m]
    for _ in range(4):
        rotate_square(m)
    assert m == orig

print('N018: 所有本章断言通过')

N018: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 实现逆时针螺旋。

**练习 2：** 解释非方阵不能套用原地旋转代码。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（逆时针螺旋）**：思路方向——整体框架（四边界、收缩、防重复检查）原样保留，只把四条边的行走顺序和方向换掉：改成左列向下、底行向右、右列向上、顶行向左。提示：最省事的做法是把顶行循环换成"左列从上到下"，其余三边依次类推，两个防重复的 if 检查要跟着新的边序调整；手算例子用 `[[1,2,3],[4,5,6]]`，逆时针应得 `[1,4,6,5,3,2]`，先自己推一遍再写代码。
- **练习 2（解释非方阵不能原地旋转）**：思路方向——从"形状"入手：2×3 矩阵旋转后应该是 3×2，行数列数互换了，而原地算法的交换操作永远在"第 i 行第 j 列 ↔ 第 j 行第 i 列"之间进行。提示：非方阵时 j 可能超过行数（比如 2 行的矩阵没有第 2 行），`matrix[j][i]` 直接越界；就算你先扩容形状，也不再是 O(1) 辅助空间。结论是：非方阵要么开新矩阵，要么先转置形状再想办法，硬套原代码必然出错。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def spiral_order(matrix):
    if not matrix or not matrix[0]:
        return []
    top, bottom, left, right = (0, len(matrix) - 1, 0, len(matrix[0]) - 1)
    out = []
    while top <= bottom and left <= right:
        for j in range(left, right + 1):
            out.append(matrix[top][j])
        top += 1
        for i in range(top, bottom + 1):
            out.append(matrix[i][right])
        right -= 1
        if top <= bottom:
            for j in range(right, left - 1, -1):
                out.append(matrix[bottom][j])
            bottom -= 1
        if left <= right:
            for i in range(bottom, top - 1, -1):
                out.append(matrix[i][left])
            left += 1
    return out

def rotate_square(matrix):
    n = len(matrix)
    if any((len(row) != n for row in matrix)):
        raise ValueError('requires square matrix')
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = (matrix[j][i], matrix[i][j])
    for row in matrix:
        row.reverse()

# 示例与回归测试
assert spiral_order([[1, 2, 3], [4, 5, 6]]) == [1, 2, 3, 6, 5, 4]

assert spiral_order([[1, 2, 3]]) == [1, 2, 3]

assert spiral_order([[1], [2], [3]]) == [1, 2, 3]

assert spiral_order([]) == []

for n in range(5):
    m = [[i * n + j for j in range(n)] for i in range(n)]
    orig = [r[:] for r in m]
    for _ in range(4):
        rotate_square(m)
    assert m == orig

print('N018: 所有本章断言通过')

N018: 所有本章断言通过


## 11. 代表题与迁移

- **54. Spiral Matrix（螺旋矩阵）**：这题就是 `spiral_order` 的原型——练的是"四边界收缩 + 单行单列防重复"这套模拟功夫。
- **48. Rotate Image（旋转图像）**：这题就是 `rotate_square` 的原型——练的是"旋转 = 转置 + 每行反转"这个坐标分解，以及 O(1) 空间的原地交换写法。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。